# Положение относительно праздников — Pilot Model v5

Цель: предсказать `boardings` для `(route, date, hour)`. Контроль — tuned XGBoost + C+W + weighted D с half-life 180 дней. Сохраняются параметры, preprocessing, postprocessing, route 5 → 0 и folds Jan–Apr → May–Jun, Jan–Jun → Jul–Aug, Jan–Aug → Sep–Oct.

Текущие calendar flags описывают статус самого дня. Новые признаки описывают положение даты относительно официального праздника или перенесённого выходного. Спрос может измениться до праздника из-за подготовки и поездок, а после него — из-за возвращения к обычному расписанию.

Источником служит только `calendar_2025_ml.csv`. Событие — `is_official_holiday OR is_transferred_day_off`; перед каждым fold проверяется `known_from <= forecast origin` для всех дат календаря с указанной датой публикации. Детерминированные выходные вычисляются из календаря без target. Для train, validation и возможного forecast действует одно определение.

Группа 1: ближайший праздник вперёд и назад, категории `0`…`7` и `>7`. Ограничение снижает влияние далёких событий и исключает бесконечные расстояния. У границ 2025 года неизвестное событие за пределами источника также получает `>7`. Группа 2: точные 1 день и окно 1–3 дня до/после события. Группа 3: дни внутри последовательности минимум из трёх подряд `is_day_off`. Три группы проверяются отдельно от чистого v5, чтобы оценить вклад каждого представления без смешения признаков.

Основной критерий — pooled WAPE по трём историческим folds. `SUPPORTED` требует улучшения pooled WAPE и победы хотя бы в двух folds; `MIXED` — pooled выигрыш на одном fold; иначе `REJECTED`. Nov–Dec leaderboard не участвует в выборе: это другой горизонт, а повторный подбор по нему создаёт риск переобучения решения.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ml/src/holiday_distance_experiment.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Project root not found')
sys.path.insert(0, str(ROOT / 'ml/src'))
from holiday_distance_experiment import run_experiment
result = run_experiment()


In [ ]:
fold_names = [fold['name'] for fold in result['folds']]
rows = []
for name, summary in result['summaries'].items():
    folds = result['fold_results'][name]
    comparison = result['comparisons_vs_pilot_v5'].get(name, {})
    rows.append({'configuration': name, **{fold_names[i]: fold['wape'] for i, fold in enumerate(folds)},
                 'pooled_wape': summary['pooled_wape'], 'delta_wape': comparison.get('delta_pooled_wape', 0),
                 'pooled_ae': summary['pooled_absolute_error'], 'wape_score': 1-summary['pooled_wape'],
                 'fit_seconds': sum(fold['train_seconds'] for fold in folds),
                 'predict_seconds': sum(fold['inference_seconds'] for fold in folds),
                 'decision': comparison.get('decision', 'REFERENCE')})
display(pd.DataFrame(rows))
display(pd.DataFrame({name: {route: metric['wape'] for route, metric in summary['per_route'].items()}
                      for name, summary in result['summaries'].items()}))
display(pd.DataFrame({name: comparison['route_delta_pooled_wape']
                      for name, comparison in result['comparisons_vs_pilot_v5'].items()}))


Полные fold и per-route метрики, время fit/predict, параметры, версии и SHA-256 источников сохраняются в `ml/experiments/results/holiday_distance_experiment.json`. Исполнение notebook повторно обучает модели; для просмотра уже полученного результата можно открыть JSON.